# Getting Started with Automated-LLM-Probes

This notebook shows a minimal trial run.

**Prerequisites**
```bash
pip install -r requirements.txt
source /Users/daweiwang/.config/llm_api_keys.sh
```

## Configs

In [ ]:
from __future__ import annotations
import os
from pathlib import Path

# Make sure keys are loaded
assert os.environ.get("OPENAI_API_KEY") or os.environ.get("ANTHROPIC_API_KEY"), \
       "Source your keys first: source /Users/daweiwang/.config/llm_api_keys.sh"

DATA_ROOT = Path("data")
print("Keys present for:", [k for k in ["OPENAI_API_KEY","ANTHROPIC_API_KEY","XAI_API_KEY","OPENROUTER_API_KEY"] if os.environ.get(k)])

## Models that are ready

In [ ]:
from automated_llm_probes import load_models, ready_models

# (If the package is not installed, the functions live in the local file)
import importlib.util
spec = importlib.util.spec_from_file_location("alp", "automated-llm-probes.py")
alp = importlib.util.module_from_spec(spec)
spec.loader.exec_module(alp)

models = alp.ready_models()
print(f"Ready models: {len(models)}")
for m in models:
    print(f"  {m['name']:20s} {m['api']:12s} {m['model_id']}")

## Tiny trial collection (DAT, 2 responses)

In [ ]:
# Restrict to a couple of models for a fast trial
trial_models = [m for m in models if m["name"] in {"GPT-4o-mini", "Claude Haiku 4.5"}][:2]
if not trial_models:
    trial_models = models[:1]   # fallback to whatever is ready

print("Will collect:", [m["name"] for m in trial_models])

# Run the collection (requires automated-intelligence-tests)
alp.collect("DAT", models=trial_models, n_per_model=2)

## Inspect the pickles that were written

In [ ]:
import pickle
from pathlib import Path

pickles = list(Path("data").rglob("*.pickle"))
print(f"Found {len(pickles)} pickles")
if pickles:
    with open(pickles[0], "rb") as f:
        row = pickle.load(f)
    print("Example row keys:", list(row.keys()))
    print("raw response preview:", (row.get("raw") or "")[:200])

## Parse & merge

In [ ]:
alp.parse_and_merge("DAT")

# Show the resulting CSV if it exists
csv_path = Path("data/dat.csv")
if csv_path.exists():
    import pandas as pd
    df = pd.read_csv(csv_path)
    display(df.head())